## Pandas 집계

### 집계
- 계산, 통계, 그룹핑으로 원데이터에서 수치적으로 필요한 값을 재가공

In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display, Markdown

orders = pd.DataFrame({
    "order_id": [1, 2, 3, 4, 5, 6],
    "customer": ["홍길동", "김철수", "이영희", "홍길동", "박민수", "김철수"],
    "product_id": ["P001", "P002", "P001", "P003", "P002", "P003"],
    "quantity": [2, 1, 3, 1, 2, 4],
    "unit_price": [30000, 15000, 30000, 8000, 15000, 8000],
    "order_date": [
        "2026-09-01",
        "2026-09-01",
        "2026-09-02",
        "2026-09-02",
        "2026-09-03",
        "2026-09-03"
    ]
})

week1 = pd.DataFrame({
    "order_id": [1, 2],
    "customer": ["홍길동", "김철수"],
    "amount": [60000, 15000]
})

week2 = pd.DataFrame({
    "order_id": [3, 4],
    "customer": ["이영희", "박민수"],
    "amount": [90000, 30000]
})

products = pd.DataFrame({
    "product_id": ["P001", "P002", "P003"],
    "product_name": ["키보드", "마우스", "USB"],
    "category": ["주변기기", "주변기기", "저장장치"]
})
products.loc[3] = {'product_id' : 'P004', 'product_name' : '모니터', 'category' : '디스플레이'}

In [ ]:
# 파생컬럼 생성
## 단가 * 수량 -> 주문금액 컬럼
orders['total_price'] = orders['quantity'] * orders['unit_price']
display(orders)

#### 조건을 이용한 컬럼 생성
- 조건연산자(<,>,=)와 논리 연산자(&,|,~) 등 사용해서 필요한 조건에 맞는 데이터만 필터링
- 필터링 후 파생컬럼 생성

In [ ]:
orders['customer_type'] = '일반'

orders

In [ ]:
condition = orders['total_price'] >= 50000
orders.loc[condition]
# 둘다 같은 의미
orders.loc[orders['total_price']>= 50000]

In [ ]:
orders.loc[orders['total_price']>= 50000, 'customer_type'] = 'vip'
orders

#### apply() 함수
- apply() 함수 내에 사용할 사용자 정의 함수 생성
- 학점 등 여러 구분으로 나눠지는 도메인 값의 컬럼을 사용할 때 유용

In [ ]:
def get_order_grade(total_price) :
    if total_price >= 80000 :
        return 'A'
    elif total_price >= 30000 :
        return 'B'
    else : 
        return 'C'

get_order_grade(90000)

In [ ]:
# total_price 컬럼의 값을 이용해 get_order_grade() 함수를 각 행별로 호출해서 grade 컬럼을 생성
orders['grade'] = orders['total_price'].apply(get_order_grade)
orders

#### 정렬

In [ ]:
# DB의 order by절과 동일
#by= ~~ : 정렬할 컬럼
# ascending False = 내림차순 / True = 오름차순
orders.sort_values(by='total_price', ascending=False)

#### CONCAT

In [ ]:
# 데이터 합치기
display(pd.concat([week1, week2])) 
# 일반적으로 가장 많이 사용하는 방법
display(pd.concat([week1,week2], ignore_index=True))

#### Merge

In [ ]:
# DB Join과 동일
pd.merge(orders, products, on='product_id', how='right')

#### 그룹핑

In [ ]:
# SQL의 Group by절과 동일
orders.groupby('customer')

# reset_index() -> 데이터프레임으로 변환
orders.groupby('customer')['total_price'].sum().reset_index()

In [ ]:
# 여러 통계를 한번에 계산
# std = 표준편차
orders.groupby('customer')['total_price'].agg(
    ['count','sum','mean','max','min','std']
).to_csv('../data/고객별_구매통계.csv', encoding='utf-8')